# Final Merchant Recommendations

This notebook presents the final merchant recommendations for BNPL onboarding.

The ranking methodology was developed and validated in `ranking_summary.ipynb`. This notebook focuses on the final decision outputs: validating the reliability of the selected merchants, presenting the final Top 100 merchants, and identifying the Top 10 merchants within each industry segment.

The final ranking combines:

- Merchant value
- Customer strength
- Revenue growth
- Revenue stability
- Market / regional context
- Fraud-risk safety

The baseline final score uses the validated weighting framework developed in the ranking methodology notebook.

## 1. Load Final Ranking Base

In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()

if (cwd / "member5_ranking").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "member5_ranking").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the repository root containing member5_ranking."
    )

RANKING_PATH = (
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "ranking_analysis_base.csv"
)

ranking_df = pd.read_csv(RANKING_PATH)

print("Shape:", ranking_df.shape)
print("Unique merchants:", ranking_df["merchant_abn"].nunique())

ranking_df.head()

Shape: (4026, 85)
Unique merchants: 4026


,merchant_abn,merchant_name,merchant_category,merchant_pricing_level,merchant_take_rate_pct,has_merchant_master_record,total_transactions,total_revenue,avg_transaction_value,unique_consumers,...,rank_change_after_fraud,fraud_5_score,fraud_5_rank,fraud_10_score,fraud_10_rank,fraud_15_score,fraud_15_rank,fraud_weight_best_rank,fraud_weight_worst_rank,fraud_weight_rank_range
0,10023283211,Felis Limited,"furniture, home furnishings and equipment shop...",E,0.18,True,3261,703277.711451,215.663205,3032,...,54,57.834749,1534,57.697612,1510,57.560474,1479,1479,1534,55
1,10142254217,Arcu Ac Orci Corporation,"cable, satellite, and other pay television and...",B,4.22,True,3036,118356.146073,38.984238,2849,...,-19,61.068892,1340,60.247282,1355,59.425671,1365,1340,1365,25
2,10165489824,Nunc Sed Company,"jewelry, watch, clock, and silverware shops",B,4.40,True,5,56180.473857,11236.094771,5,...,213,25.759912,3511,28.881561,3391,32.003211,3254,3254,3511,257
3,10187291046,Ultricies Dignissim Lacus Foundation,"watch, clock, and jewelry repair shops",B,3.29,True,336,39693.730387,118.136102,335,...,74,42.076145,2538,42.741455,2491,43.406765,2441,2441,2538,97
4,10192359162,Enim Condimentum PC,"music shops - musical instruments, pianos, and...",A,6.33,True,385,177980.505456,462.287027,383,...,-128,58.536868,1495,56.713603,1561,54.890338,1627,1495,1627,132


## 2. Final Ranking Reliability Audit

Before presenting the final recommendations, the Top 100 merchants are checked for three potential reliability concerns:

1. whether growth estimates are based on insufficient transaction history,
2. whether external Census / SEIFA / ATO coverage is adequate, and
3. whether fraud-risk evidence is sufficiently available.

These checks do not directly change the ranking. They are used to assess how much confidence should be placed in the final recommendations.

In [2]:
final_top100_df = (
    ranking_df
    .sort_values("final_rank")
    .head(100)
    .copy()
)

print("Final Top 100 merchants:", len(final_top100_df))

Final Top 100 merchants: 100


In [3]:
growth_reliability = (
    final_top100_df["low_sample_growth_estimate"]
    .value_counts(dropna=False)
    .rename_axis("low_sample_growth_estimate")
    .to_frame("merchant_count")
)

growth_reliability

,merchant_count
low_sample_growth_estimate,
False,100


In [4]:
low_sample_count = final_top100_df["low_sample_growth_estimate"].fillna(False).sum()

print("Low-sample growth merchants:", low_sample_count)
print("Share of Top 100:", low_sample_count / 100)

Low-sample growth merchants: 0
Share of Top 100: 0.0


In [5]:
external_coverage_summary = (
    final_top100_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

external_coverage_summary

count    100.000000
mean       0.807774
std        0.002932
min        0.800461
25%        0.805966
50%        0.807791
75%        0.809167
max        0.815884
Name: regional_data_coverage_rate_all_sources_by_count, dtype: float64

In [6]:
coverage_checks = pd.Series({
    "coverage_below_90pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.90
        ).sum(),

    "coverage_below_80pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.80
        ).sum(),

    "coverage_below_70pct":
        (
            final_top100_df[
                "regional_data_coverage_rate_all_sources_by_count"
            ] < 0.70
        ).sum(),
})

coverage_checks

coverage_below_90pct    100
coverage_below_80pct      0
coverage_below_70pct      0
dtype: int64

In [7]:
overall_external_coverage = (
    ranking_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

top100_external_coverage = (
    final_top100_df[
        "regional_data_coverage_rate_all_sources_by_count"
    ]
    .describe()
)

coverage_comparison = pd.DataFrame({
    "all_eligible_merchants": overall_external_coverage,
    "final_top_100": top100_external_coverage,
})

coverage_comparison

,all_eligible_merchants,final_top_100
count,4026.000000,100.000000
mean,0.806718,0.807774
std,0.054179,0.002932
min,0.000000,0.800461
25%,0.796875,0.805966
50%,0.807871,0.807791
75%,0.819749,0.809167
max,1.000000,0.815884


In [8]:
overall_mean_coverage = ranking_df[
    "regional_data_coverage_rate_all_sources_by_count"
].mean()

top100_mean_coverage = final_top100_df[
    "regional_data_coverage_rate_all_sources_by_count"
].mean()

print("All eligible merchants mean coverage:", overall_mean_coverage)
print("Top 100 mean coverage:", top100_mean_coverage)
print("Difference:", top100_mean_coverage - overall_mean_coverage)

All eligible merchants mean coverage: 0.806718209745659
Top 100 mean coverage: 0.8077740362121092
Difference: 0.0010558264664501937


External-data coverage is highly consistent among the final Top 100 merchants. Their mean joint coverage across Census, SEIFA, and ATO sources is 80.81%, which is very close to the 80.67% average across all eligible merchants.

This suggests that the final recommendations are not disproportionately driven by merchants with poor external-data coverage.

In [9]:
fraud_evidence_summary = (
    final_top100_df["fraud_evidence_status"]
    .value_counts(dropna=False)
    .to_frame("merchant_count")
)

fraud_evidence_summary

,merchant_count
fraud_evidence_status,
consumer_and_knn,100


In [10]:
print(
    "Top 100 with KNN merchant-risk scores:",
    final_top100_df[
        "has_knn_merchant_risk_information"
    ].fillna(False).sum()
)

print(
    "Top 100 with consumer fraud information:",
    final_top100_df[
        "has_consumer_risk_information"
    ].fillna(False).sum()
)

Top 100 with KNN merchant-risk scores: 100
Top 100 with consumer fraud information: 100


### Reliability Audit Interpretation

The final Top 100 shows strong reliability across the main diagnostic checks.

None of the selected merchants is flagged as having a low-sample growth estimate, indicating that the growth component is not being driven by merchants with insufficient transaction history.

External-data coverage is also consistent. The Top 100 has an average joint Census, SEIFA, and ATO coverage of 80.81%, compared with 80.67% across all eligible merchants, suggesting that the final recommendations are not disproportionately affected by poor external-data coverage.

All Top 100 merchants have consumer-level fraud information and a KNN merchant-risk score. The KNN score is trained from the 61 directly observed merchants and generalised to all eligible merchants; direct observations remain diagnostics rather than a manually inserted score.

## 3. Locked Final Scoring Framework

Following feature construction, business-weight sensitivity analysis, fraud integration, fraud-weight sensitivity analysis, and reliability checks, the final ranking framework is fixed before producing the recommendation lists.

The final score is:

\[
\text{Final Score}
=
0.90 \times \text{Business Score}
+
0.10 \times \text{Fraud-Risk Safety Score}
\]

where:

\[
\text{Business Score}
=
0.30 \times \text{Value}
+
0.25 \times \text{Customer Strength}
+
0.20 \times \text{Growth}
+
0.15 \times \text{Stability}
+
0.10 \times \text{Market Context}
\]

This is equivalent to the following effective final weights:

- **Merchant Value: 27%**
- **Customer Strength: 22.5%**
- **Growth: 18%**
- **Stability: 13.5%**
- **Market / Regional Context: 9%**
- **Fraud-Risk Safety: 10%**

The weighting framework prioritises directly observed commercial performance while retaining supporting signals for future growth, consistency, socioeconomic context, and fraud risk.

Reliability checks indicate that:

- none of the final Top 100 merchants has a low-sample growth estimate;
- external Census, SEIFA, and ATO coverage among the Top 100 is consistent with the wider eligible merchant pool; and
- all Top 100 merchants have consumer-level fraud information, with a subset also supported by direct merchant-level fraud evidence.

The scoring framework is therefore treated as fixed for the final recommendation stage.

In [11]:
required_final_columns = [
    "final_score",
    "final_rank",
    "balanced_score",
    "risk_safety_score",
    "value_score",
    "customer_score",
    "growth_score",
    "stability_score",
    "market_score",
]

missing_final_columns = [
    col for col in required_final_columns
    if col not in ranking_df.columns
]

print("Missing required final columns:", missing_final_columns)

Missing required final columns: []


## 4. Final Top 100 Merchants

The final Top 100 merchants are selected using the locked final scoring framework.

These merchants represent the strongest overall onboarding candidates after considering commercial value, customer strength, growth, stability, market context, and fraud-risk safety.

In [12]:
final_top100 = (
    ranking_df
    .sort_values("final_rank")
    .head(100)
    .copy()
)

final_top100[
    [
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
        "value_score",
        "customer_score",
        "growth_score",
        "stability_score",
        "market_score",
    ]
].head(20)

,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score,value_score,customer_score,growth_score,stability_score,market_score
1301,1,38090089066,Interdum Feugiat Sed Inc.,"furniture, home furnishings and equipment shop...",85.186272,86.800764,70.655847,98.683557,99.254844,61.607365,93.928838,59.711873
803,2,27326652377,Tellus Aenean Corporation,"music shops - musical instruments, pianos, and...",84.893301,86.253316,72.653174,99.403875,89.195231,75.267479,86.165713,61.549925
3614,3,90568944804,Diam Eu Dolor LLC,tent and awning shops,84.882118,87.168863,64.301410,99.230005,93.169399,75.167952,82.483205,67.014406
3369,4,84703983173,Amet Consulting,"computer programming , data processing, and in...",84.765184,85.765124,75.765729,96.572280,99.006458,68.847972,89.524757,48.435171
3443,5,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops",84.661574,85.650047,75.765319,99.925484,99.975161,55.113212,95.172929,53.800298
1401,6,40515428545,Elit Sed Consequat Associates,artist supply and craft shops,84.515082,84.659810,83.212536,99.652260,94.585196,50.684250,94.700174,67.759563
348,7,17488304283,Posuere Cubilia Curae Corporation,"cable, satellite, and other pay television and...",84.462702,85.080227,78.904983,97.913562,98.559364,64.170192,93.480965,42.101341
1798,8,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers",84.412749,85.552242,74.157318,99.056135,98.782911,60.014929,94.575765,49.503229
3453,9,86772484982,Posuere Cubilia Curae LLC,"digital goods: books, movies, music",84.292045,85.785250,70.853196,95.032290,95.355191,59.442647,98.283155,68.057625
3881,10,96680767841,Ornare Limited,motor vehicle supplies and new parts,84.282740,85.723766,71.313509,99.850969,98.435171,63.772083,90.097039,48.907104


In [13]:
top20_inspection = final_top100[
    [
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
        "value_score",
        "customer_score",
        "growth_score",
        "stability_score",
        "market_score",
        "fraud_evidence_status",
    ]
].head(20).copy()

top20_inspection.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "final_top20_inspection.csv",
    index=False
)

top20_inspection

,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score,value_score,customer_score,growth_score,stability_score,market_score,fraud_evidence_status
1301,1,38090089066,Interdum Feugiat Sed Inc.,"furniture, home furnishings and equipment shop...",85.186272,86.800764,70.655847,98.683557,99.254844,61.607365,93.928838,59.711873,consumer_and_knn
803,2,27326652377,Tellus Aenean Corporation,"music shops - musical instruments, pianos, and...",84.893301,86.253316,72.653174,99.403875,89.195231,75.267479,86.165713,61.549925,consumer_and_knn
3614,3,90568944804,Diam Eu Dolor LLC,tent and awning shops,84.882118,87.168863,64.301410,99.230005,93.169399,75.167952,82.483205,67.014406,consumer_and_knn
3369,4,84703983173,Amet Consulting,"computer programming , data processing, and in...",84.765184,85.765124,75.765729,96.572280,99.006458,68.847972,89.524757,48.435171,consumer_and_knn
3443,5,86578477987,Leo In Consulting,"watch, clock, and jewelry repair shops",84.661574,85.650047,75.765319,99.925484,99.975161,55.113212,95.172929,53.800298,consumer_and_knn
1401,6,40515428545,Elit Sed Consequat Associates,artist supply and craft shops,84.515082,84.659810,83.212536,99.652260,94.585196,50.684250,94.700174,67.759563,consumer_and_knn
348,7,17488304283,Posuere Cubilia Curae Corporation,"cable, satellite, and other pay television and...",84.462702,85.080227,78.904983,97.913562,98.559364,64.170192,93.480965,42.101341,consumer_and_knn
1798,8,49212265466,Auctor Company,"florists supplies, nursery stock, and flowers",84.412749,85.552242,74.157318,99.056135,98.782911,60.014929,94.575765,49.503229,consumer_and_knn
3453,9,86772484982,Posuere Cubilia Curae LLC,"digital goods: books, movies, music",84.292045,85.785250,70.853196,95.032290,95.355191,59.442647,98.283155,68.057625,consumer_and_knn
3881,10,96680767841,Ornare Limited,motor vehicle supplies and new parts,84.282740,85.723766,71.313509,99.850969,98.435171,63.772083,90.097039,48.907104,consumer_and_knn


In [14]:
final_top20 = (
    final_top100
    .head(20)
    .copy()
)

print("Final Top 20 merchants:", len(final_top20))

Final Top 20 merchants: 20


In [15]:
top20_score_summary = (
    final_top20[
        [
            "final_score",
            "balanced_score",
            "risk_safety_score",
            "value_score",
            "customer_score",
            "growth_score",
            "stability_score",
            "market_score",
        ]
    ]
    .describe()
    .T
)

top20_score_summary

,count,mean,std,min,25%,50%,75%,max
final_score,20.0,84.276541,0.431230,83.740287,83.896764,84.217723,84.551705,85.186272
balanced_score,20.0,85.403392,0.707057,84.426294,84.899718,85.226394,85.734106,87.168863
risk_safety_score,20.0,74.134885,3.997372,64.301410,72.324159,73.506120,75.765421,83.212536
value_score,20.0,98.361898,1.570648,95.032290,97.540984,98.969200,99.484600,99.975161
customer_score,20.0,96.493418,4.144790,83.457526,95.162692,98.484848,99.068554,99.975161
growth_score,20.0,61.686987,6.621026,50.684250,57.091316,60.811147,64.058223,75.267479
stability_score,20.0,91.255287,4.052649,82.483205,89.879323,91.714357,94.090570,98.283155
market_score,20.0,57.457774,9.798435,42.101341,48.944362,54.992548,67.200695,73.571783


In [16]:
score_dimensions = [
    "value_score",
    "customer_score",
    "growth_score",
    "stability_score",
    "market_score",
    "risk_safety_score",
]

top20_weakest_dimension = final_top20[
    ["final_rank", "merchant_name"] + score_dimensions
].copy()

top20_weakest_dimension["weakest_dimension"] = (
    top20_weakest_dimension[score_dimensions].idxmin(axis=1)
)

top20_weakest_dimension["weakest_score"] = (
    top20_weakest_dimension[score_dimensions].min(axis=1)
)

top20_weakest_dimension[
    [
        "final_rank",
        "merchant_name",
        "weakest_dimension",
        "weakest_score",
    ]
]

,final_rank,merchant_name,weakest_dimension,weakest_score
1301,1,Interdum Feugiat Sed Inc.,market_score,59.711873
803,2,Tellus Aenean Corporation,market_score,61.549925
3614,3,Diam Eu Dolor LLC,risk_safety_score,64.301410
3369,4,Amet Consulting,market_score,48.435171
3443,5,Leo In Consulting,market_score,53.800298
1401,6,Elit Sed Consequat Associates,growth_score,50.684250
348,7,Posuere Cubilia Curae Corporation,market_score,42.101341
1798,8,Auctor Company,market_score,49.503229
3453,9,Posuere Cubilia Curae LLC,growth_score,59.442647
3881,10,Ornare Limited,market_score,48.907104


### Top 20 Profile Check

The Top 20 merchants are consistently strong in the core commercial dimensions of merchant value, customer strength, and revenue stability.

Variation is mainly concentrated in growth, market context, and fraud-risk safety. No Top 20 merchant exhibits an extreme weakness in the core commercial dimensions, suggesting that the highest-ranked merchants are supported by broad business strength rather than a single dominant metric.

In [17]:
final_top100.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "final_top_100.csv",
    index=False
)

print("Final Top 100 exported successfully.")

Final Top 100 exported successfully.


## 5. Segment-Level Top 10 Recommendations

To complement the overall Top 100 ranking, merchants are also compared within their broader industry segments.

The same locked final score is used for all segments. This preserves consistency across the recommendation framework while allowing strong merchants in smaller or structurally different industries to be identified.

For each segment, the Top 10 merchants are selected according to their final score.

In [18]:
segment_candidates = [
    col for col in ranking_df.columns
    if "segment" in col.lower()
    or "industry" in col.lower()
    or "group" in col.lower()
]

segment_candidates

[]

In [19]:
MAPPING_PATH = (
    PROJECT_ROOT
    / "member3_industry_growth"
    / "results"
    / "category_to_group_mapping.csv"
)

segment_mapping = pd.read_csv(MAPPING_PATH)

print("Mapping shape:", segment_mapping.shape)
segment_mapping.head()

Mapping shape: (25, 2)


,merchant_category,industry_group
0,"antique shops - sales, repairs, and restoratio...","Art, Gifts, Jewellery & Fashion"
1,art dealers and galleries,"Art, Gifts, Jewellery & Fashion"
2,artist supply and craft shops,"Creative, Books & Leisure"
3,bicycle shops - sales and service,"Mobility, Health & Specialist Services"
4,"books, periodicals, and newspapers","Creative, Books & Leisure"


In [20]:
segment_mapping.columns.tolist()

['merchant_category', 'industry_group']

In [21]:
ranking_with_segment = ranking_df.merge(
    segment_mapping,
    on="merchant_category",
    how="left",
    validate="many_to_one",
)

print("Before merge:", len(ranking_df))
print("After merge:", len(ranking_with_segment))
print(
    "Missing industry group:",
    ranking_with_segment["industry_group"].isna().sum()
)

ranking_with_segment["industry_group"].value_counts()

Before merge: 4026
After merge: 4026
Missing industry group: 0


industry_group
Digital, Technology & Communications      867
Home, Garden & Living                     827
Creative, Books & Leisure                 827
Mobility, Health & Specialist Services    806
Art, Gifts, Jewellery & Fashion           699
Name: count, dtype: int64

In [22]:
segment_top10 = (
    ranking_with_segment
    .sort_values(
        ["industry_group", "final_score"],
        ascending=[True, False]
    )
    .groupby("industry_group", group_keys=False)
    .head(10)
    .copy()
)

segment_top10["segment_rank"] = (
    segment_top10
    .groupby("industry_group")["final_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

segment_top10[
    [
        "industry_group",
        "segment_rank",
        "final_rank",
        "merchant_abn",
        "merchant_name",
        "merchant_category",
        "final_score",
        "balanced_score",
        "risk_safety_score",
    ]
].sort_values(
    ["industry_group", "segment_rank"]
)

,industry_group,segment_rank,final_rank,merchant_abn,merchant_name,merchant_category,final_score,balanced_score,risk_safety_score
2638,"Art, Gifts, Jewellery & Fashion",1,18,68559320474,Aliquam Auctor Associates,"antique shops - sales, repairs, and restoratio...",83.803132,84.968060,73.318783
3796,"Art, Gifts, Jewellery & Fashion",2,24,94493496784,Dictum Phasellus In Institute,"gift, card, novelty, and souvenir shops",83.651983,84.910756,72.323021
2293,"Art, Gifts, Jewellery & Fashion",3,32,60956456424,Ultricies Dignissim LLP,"gift, card, novelty, and souvenir shops",83.338210,84.398617,73.794548
3139,"Art, Gifts, Jewellery & Fashion",4,34,79417999332,Phasellus At Company,"gift, card, novelty, and souvenir shops",83.210306,84.712139,69.693808
3957,"Art, Gifts, Jewellery & Fashion",5,37,98314397036,Lobortis Augue Industries,"gift, card, novelty, and souvenir shops",83.145643,84.544309,70.557648
1637,"Art, Gifts, Jewellery & Fashion",6,38,45629217853,Lacus Consulting,"gift, card, novelty, and souvenir shops",83.139651,83.962811,75.731205
65,"Art, Gifts, Jewellery & Fashion",7,52,11439466003,Blandit At LLC,shoe shops,82.621604,83.065984,78.622180
3450,"Art, Gifts, Jewellery & Fashion",8,53,86710922099,Ac Urna Consulting,art dealers and galleries,82.607560,82.541010,83.206515
2512,"Art, Gifts, Jewellery & Fashion",9,54,66079287213,Hendrerit Corporation,"gift, card, novelty, and souvenir shops",82.460803,84.162466,67.145834
3753,"Art, Gifts, Jewellery & Fashion",10,57,93558142492,Dolor Quisque Inc.,shoe shops,82.392939,82.680867,79.801583


In [23]:
print("Total selected merchants:", len(segment_top10))

segment_top10.groupby(
    "industry_group"
).size()

Total selected merchants: 50


industry_group
Art, Gifts, Jewellery & Fashion           10
Creative, Books & Leisure                 10
Digital, Technology & Communications      10
Home, Garden & Living                     10
Mobility, Health & Specialist Services    10
dtype: int64

In [24]:
print(
    "Unique merchants in segment Top 10:",
    segment_top10["merchant_abn"].nunique()
)

Unique merchants in segment Top 10: 50


### Segment-Level Summary

The segment-level recommendations use the same locked final score as the overall ranking.

The summary below compares the five industry groups using the average final score, average fraud-risk safety, and the range of overall ranks represented within each segment Top 10.

In [25]:
segment_summary = (
    segment_top10
    .groupby("industry_group")
    .agg(
        merchants_selected=("merchant_abn", "count"),
        mean_final_score=("final_score", "mean"),
        mean_balanced_score=("balanced_score", "mean"),
        mean_risk_safety_score=("risk_safety_score", "mean"),
        best_overall_rank=("final_rank", "min"),
        worst_overall_rank=("final_rank", "max"),
    )
    .sort_values("mean_final_score", ascending=False)
)

segment_summary

,merchants_selected,mean_final_score,mean_balanced_score,mean_risk_safety_score,best_overall_rank,worst_overall_rank
industry_group,,,,,,
"Home, Garden & Living",10,83.966235,85.143734,73.368743,1,33
"Digital, Technology & Communications",10,83.781812,84.755847,75.015497,4,44
"Mobility, Health & Specialist Services",10,83.650043,84.874692,72.628203,5,49
"Creative, Books & Leisure",10,83.446161,84.631769,72.775683,2,73
"Art, Gifts, Jewellery & Fashion",10,83.037183,83.994702,74.419513,18,57


In [26]:
segment_score_spread = (
    segment_top10
    .groupby("industry_group")["final_score"]
    .agg(["min", "median", "max"])
)

segment_score_spread["score_range"] = (
    segment_score_spread["max"]
    - segment_score_spread["min"]
)

segment_score_spread

,min,median,max,score_range
industry_group,,,,
"Art, Gifts, Jewellery & Fashion",82.392939,83.142647,83.803132,1.410193
"Creative, Books & Leisure",82.081251,83.540952,84.893301,2.812050
"Digital, Technology & Communications",82.976696,83.723417,84.765184,1.788489
"Home, Garden & Living",83.274555,83.802259,85.186272,1.911718
"Mobility, Health & Specialist Services",82.785708,83.776304,84.661574,1.875867


### Segment-Level Interpretation

The five industry segments show broadly similar performance among their Top 10 merchants. Mean final scores range from approximately 81.1 to 82.0, suggesting that the final scoring framework does not strongly favour a single industry group.

Within each segment, the Top 10 scores are also relatively concentrated, with score ranges of approximately 1.5 to 3.5 points.

All 50 segment-level Top 10 merchants also fall within the overall Top 100 ranking. Therefore, the segment-level recommendations do not introduce weaker merchants solely to achieve industry representation. Instead, they provide an industry-specific view of merchants that are already strong overall candidates.

In [27]:
segment_top10.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "segment_top_10.csv",
    index=False
)

segment_summary.to_csv(
    PROJECT_ROOT
    / "member5_ranking"
    / "results"
    / "segment_summary.csv"
)

print("Segment Top 10 and segment summary exported successfully.")

Segment Top 10 and segment summary exported successfully.